<!-- ## Pipeline AI Engineer dalam merancang chatbot edukasi IPA Sekolah Dasar kelas 5 -->

<!-- - **TensorFlow Functional API** — klasifikasi topik pertanyaan
- **Custom Layer + Custom Loss + Custom Callback** — komponen DL kustom
- **TiDB Vector DB** — penyimpanan & retrieval embedding (RAG)
- **BAAI/bge-m3** — model embedding multilingual (1024 dim)
- **Moderation System** — filter kata kasar (strike + cooldown)
- **Screen Time Manager** — pengingat istirahat per sesi belajar
- **FastAPI** — REST API siap produksi

---
**File yang dihasilkan**: `semantic_faq_model.keras` · `semantic_faq_savedmodel/` · `label_mapping.json` · `app.py`  
**Sebelum mulai**: isi `config.env` dengan kredensial TiDB kamu (lihat Section 8) -->


<!-- #### Tahap 1 Instalansi Library -->

<!-- #### Tahap 2 Imports Library -->

In [6]:
# %pip install tensorflow==2.15.0 \
#              keras==2.15.0 \
#              numpy==1.26.4 \
#              pandas==3.0.3 \
#              torch==2.4.0 --index-url https://download.pytorch.org/whl/cpu \
#              torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cpu \
#              sentence-transformers==3.0.1 \
#              transformers==4.41.2 \
#              scikit-learn==1.8.0 \
#              mysql-connector-python==9.7.0 \
#              python-dotenv==1.2.2 \
#              fastapi==0.136.3 \
#              uvicorn==0.48.0 \
#              nest-asyncio==1.6.0 \
#              requests==2.34.2 \

In [7]:
import csv
import json
import logging
import os
import random
import re
from datetime import datetime, timedelta

import mysql.connector
import numpy as np
import pandas as pd
import tensorflow as tf
from dotenv import load_dotenv
from mysql.connector import Error
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split

from tensorflow.keras import losses
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.layers import (
    Dropout, Dense, Embedding,
    Input, Layer, TextVectorization
)
from tensorflow.keras.models import Model

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

print("TensorFlow :", tf.__version__)
print("NumPy      :", np.__version__)

c:\Users\Erycaa\Downloads\caps\DBS-Capstone-2026\env\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


TensorFlow : 2.15.0
NumPy      : 1.26.4


<!-- Load data dan preprocessing -->

In [8]:
data = pd.read_csv(
    r'C:\Users\Erycaa\Downloads\caps\DBS-Capstone-2026\dataset\dataclean_revisi.csv'
)
print('Shape:', data.shape)
data.head(3)

Shape: (3348, 15)


,no,topik,subtopik,soal,jawaban,contoh,konteks,link_sumber,panjang_soal,panjang_jawaban,panjang_konteks,kata_soal,kata_jawaban,jenis_pertanyaan,kompleksitas
0,1,adaptasi makhluk hidup,adaptasi makhluk hidup,kemampuan makhluk hidup untuk menyesuaikan dir...,adaptasi adalah kemampuan makhluk hidup menyes...,adaptasi membantu survival makhluk hidup,adaptasi terjadi dalam waktu lama,https://www.omahbse.com/ktsp/file/sd-5_ipa_007...,77,103,33,9,13,lainnya,Sedang
1,2,adaptasi makhluk hidup,adaptasi makhluk hidup,bentuk adaptasi yang berkaitan dengan fungsi o...,adaptasi fisiologis adalah penyesuaian kerja o...,racun produksi adalah adaptasi,adaptasi fisiologi tidak terlihat,https://www.omahbse.com/ktsp/file/sd-5_ipa_007...,67,86,33,9,12,lainnya,Sedang
2,3,adaptasi makhluk hidup,adaptasi makhluk hidup,bentuk adaptasi yang berkaitan dengan bentuk t...,adaptasi morfologi adalah penyesuaian bentuk t...,warna kulit paruh kuku adalah adaptasi,adaptasi morfologi terlihat secara fisik,https://www.omahbse.com/ktsp/file/sd-5_ipa_007...,77,83,40,11,11,lainnya,Kompleks


In [9]:
# isi null dulu biar ga error waktu concat
for col in ['soal', 'jawaban', 'contoh', 'konteks']:
    data[col] = data[col].fillna('').astype(str)

# bersihin teks - lowercase, buang karakter aneh
def bersihkan(teks):
    teks = teks.lower()
    teks = re.sub(r'[^a-zA-Z0-9\s]', ' ', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks

# gabungin semua kolom teks jadi satu
data['content'] = data['soal'] + ' ' + data['jawaban'] + ' ' + data['contoh'] + ' ' + data['konteks']
data['content'] = data['content'].apply(bersihkan)

print('preprocessing beres')
data[['soal', 'content']].head(3)

preprocessing beres


,soal,content
0,kemampuan makhluk hidup untuk menyesuaikan dir...,kemampuan makhluk hidup untuk menyesuaikan dir...
1,bentuk adaptasi yang berkaitan dengan fungsi o...,bentuk adaptasi yang berkaitan dengan fungsi o...
2,bentuk adaptasi yang berkaitan dengan bentuk t...,bentuk adaptasi yang berkaitan dengan bentuk t...


In [10]:
# encode topik jadi angka
data['topik'] = data['topik'].astype('category')
data['label'] = data['topik'].cat.codes
label_map = dict(enumerate(data['topik'].cat.categories))

print(f'ada {len(label_map)} kelas topik:')
print(label_map)

ada 15 kelas topik:
{0: 'adaptasi makhluk hidup', 1: 'air', 2: 'alat pencernaan dan makanan', 3: 'alat pernapasan manusia dan hewan', 4: 'alat tubuh manusia dan hewan', 5: 'benda dan sifatnya', 6: 'bumi dan peristiwa alam', 7: 'cahaya dan sifat-sifatnya', 8: 'gaya, gerak, dan energi', 9: 'organ tubuh manusia dan hewan', 10: 'peredaran darah', 11: 'peristiwa alam', 12: 'sistem pernapasan', 13: 'sumber daya alam dan kegunaannya', 14: 'tumbuhan hijau'}


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    data['soal'], data['label'],
    test_size=0.2, random_state=42, stratify=data['label']
)

print('train:', len(X_train), '| test:', len(X_test))

train: 2678 | test: 670


In [12]:
# pakai 10000 vocab, panjang max 100 token
vectorizer = TextVectorization(max_tokens=10000, output_mode='int', output_sequence_length=100)
vectorizer.adapt(X_train)

print('vocab size:', len(vectorizer.get_vocabulary()))

2026-05-30 10:10:19,891 [WARNING] From c:\Users\Erycaa\Downloads\caps\DBS-Capstone-2026\env\Lib\site-packages\keras\src\backend.py:873: The name tf.get_default_graph is deprecated. Please use tf.compat.v1.get_default_graph instead.



2026-05-30 10:10:20,576 [WARNING] From c:\Users\Erycaa\Downloads\caps\DBS-Capstone-2026\env\Lib\site-packages\keras\src\utils\tf_utils.py:492: The name tf.ragged.RaggedTensorValue is deprecated. Please use tf.compat.v1.ragged.RaggedTensorValue instead.



vocab size: 1080


<!-- # Tahap 4 CUSTOM COMPONENTS
**1. Custom Layer**  

**2. Custom Loss**

**3. Custom Callback** -->

<!-- ## 4.1 Custom Layer: `AttentionPooling`
Attention-weighted pooling — belajar memberi bobot lebih pada token yang paling penting. -->

In [13]:
class AttentionPooling(Layer):
    """
    Custom Layer — Attention-weighted pooling.
    Setiap token mendapat skor perhatian yang dipelajari selama training,
    lalu dilakukan weighted sum. Lebih ekspresif dari GlobalAveragePooling.
    """
    def build(self, input_shape):
        self.attention_weights = self.add_weight(
            name='attention_weights', shape=(input_shape[-1],),
            initializer='glorot_uniform', trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        scores = tf.tensordot(inputs, self.attention_weights, axes=[[2], [0]])
        scores = tf.nn.softmax(scores, axis=-1)
        return tf.reduce_sum(inputs * tf.expand_dims(scores, -1), axis=1)

    def get_config(self): return super().get_config()

print("AttentionPooling selesai")


AttentionPooling selesai


<!-- ## 4B — Custom Loss: `FocalLoss`
Fokus ke sampel yang sulit diklasifikasi — cocok untuk dataset tidak seimbang. -->

In [14]:
class FocalLoss(losses.Loss):
    """
    Custom Loss — Focal Loss untuk multi-class classification.
    FL(p_t) = -alpha * (1 - p_t)^gamma * log(p_t)
    Mengurangi bobot loss sampel mudah sehingga model fokus ke sampel sulit.
    """
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        # Squeeze y_true untuk memastikan berdimensi (batch_size,) sebelum di-one_hot
        y_true = tf.squeeze(tf.cast(y_true, tf.int32))
        n  = tf.shape(y_pred)[-1]
        oh = tf.one_hot(y_true, n)
        ce = -oh * tf.math.log(y_pred + 1e-7)
        w  = self.alpha * tf.pow(1.0 - y_pred, self.gamma)
        # Kembalikan loss per-sample (shape: (batch_size,))
        return tf.reduce_sum(w * ce, axis=-1)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'gamma': self.gamma, 'alpha': self.alpha})
        return cfg

print("FocalLoss selesai")


FocalLoss selesai


<!-- ## 4C — Custom Callback: `EarlyStopOnAccuracy`
Stop training otomatis saat akurasi melewati threshold.

Buat callback yang otomatis stop training kalau accuracy udah cukup tinggi,
biar ga buang-buang waktu train terus sampe epoch selesai. -->

In [15]:
class StopKalauUdahBagus(Callback):
    # stop training otomatis kalau acc udah lewatin threshold

    def __init__(self, threshold=0.90):
        super().__init__()
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            logs = {}

        acc = logs.get('accuracy', 0)
        val = logs.get('val_accuracy', 0)
        print(f'epoch {epoch+1} -> acc: {acc:.3f}, val_acc: {val:.3f}')

        if acc >= self.threshold:
            print(f'acc udah {acc:.3f}, stop training sekarang')
            self.model.stop_training = True


print('callback siap')

callback siap


<!-- ## Bangun Model (Functional API)

Arsitektur: input teks -> vectorize -> embedding -> pooling -> dense layers -> output softmax -->

In [16]:
n_kelas = len(label_map)

inp = Input(shape=(1,), dtype=tf.string, name='input_text')
x   = vectorizer(inp)
x   = Embedding(input_dim=10_000, output_dim=128, name='embedding')(x)
x   = AttentionPooling(name='attention_pooling')(x)   # Custom Layer
x   = Dense(256, activation='relu')(x)
x   = Dropout(0.4)(x)
x   = Dense(128, activation='relu')(x)
x   = Dropout(0.3)(x)
x   = Dense(64, activation='relu', name='feature_layer')(x)
out = Dense(n_kelas, activation='softmax', name='output')(x)

model = Model(inputs=inp, outputs=out, name='chatbot_ipa_model')
model.summary()


Model: "chatbot_ipa_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_text (InputLayer)     [(None, 1)]               0         
                                                                 
 text_vectorization (TextVe  (None, 100)               0         
 ctorization)                                                    
                                                                 
 embedding (Embedding)       (None, 100, 128)          1280000   
                                                                 
 attention_pooling (Attenti  (None, 128)               128       
 onPooling)                                                      
                                                                 
 dense (Dense)               (None, 256)               33024     
                                                                 
 dropout (Dropout)           (None, 256)         

<!-- #### Perbaikan untuk bagian loss fucntion untuk bagian categoriclad cross antropy -->

In [17]:
# model.compile(
#     optimizer="adam",
#     loss="sparse_categorical_crossentropy",
#     metrics=[
#         "accuracy",
#         tf.keras.metrics.MeanAbsoluteError(name="mae")
#     ],
# )

In [18]:
# Definisikan Metrik Kustom OneHotMAE agar sesuai dengan target Side Quest (MAE <= 0.02)
class OneHotMAE(tf.keras.metrics.Metric):
    def __init__(self, name='mae', **kwargs):
        super().__init__(name=name, **kwargs)
        self.mae_tracker = tf.keras.metrics.MeanAbsoluteError()
    def update_state(self, y_true, y_pred, sample_weight=None):
        n = tf.shape(y_pred)[-1]
        y_true_oh = tf.one_hot(tf.squeeze(tf.cast(y_true, tf.int32)), n)
        self.mae_tracker.update_state(y_true_oh, y_pred, sample_weight)
    def result(self): return self.mae_tracker.result()
    def reset_state(self): self.mae_tracker.reset_state()

model.compile(
    optimizer='adam',
    loss=FocalLoss(gamma=2.0, alpha=0.25),   # Custom Loss
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy'), # Sparse Accuracy Akurat
        OneHotMAE(name='mae')                                        # MAE Kelas Riil
    ]
)
print("Compile selesai ✔ (Dengan metrik evaluasi OneHotMAE & Sparse Accuracy)")


2026-05-30 10:10:21,602 [WARNING] From c:\Users\Erycaa\Downloads\caps\DBS-Capstone-2026\env\Lib\site-packages\keras\src\optimizers\__init__.py:309: The name tf.train.Optimizer is deprecated. Please use tf.compat.v1.train.Optimizer instead.



Compile selesai ✔ (Dengan metrik evaluasi OneHotMAE & Sparse Accuracy)


<!-- ## Training -->

In [19]:
stop_callback = StopKalauUdahBagus(threshold=0.90)

history = model.fit(
    tf.constant(X_train, dtype=tf.string), y_train,
    validation_data=(tf.constant(X_test, dtype=tf.string), y_test),
    epochs=50,
    batch_size=32,
    callbacks=[stop_callback],
    verbose=0  # print dari callback aja
)

epoch 1 -> acc: 0.123, val_acc: 0.136
epoch 2 -> acc: 0.142, val_acc: 0.163
epoch 3 -> acc: 0.252, val_acc: 0.466
epoch 4 -> acc: 0.582, val_acc: 0.754
epoch 5 -> acc: 0.763, val_acc: 0.833
epoch 6 -> acc: 0.834, val_acc: 0.855
epoch 7 -> acc: 0.874, val_acc: 0.878
epoch 8 -> acc: 0.893, val_acc: 0.894
epoch 9 -> acc: 0.906, val_acc: 0.890
acc udah 0.906, stop training sekarang


In [20]:
loss, acc, mae = model.evaluate(
    tf.constant(X_test, dtype=tf.string), y_test, verbose=0
)

print('hasil di test set:')
print(f'  loss    : {loss:.4f}')
print(f'  accuracy: {acc:.4f}')
print(f'  mae     : {mae:.4f}')

hasil di test set:
  loss    : 0.0455
  accuracy: 0.8896
  mae     : 0.0269


<!-- # SIMPAN & EKSPOR MODEL  -->

In [21]:
# Format 1: .keras 
model.save('semantic_faq_model.keras')
print("semantic_faq_model.keras")

# Format 2: SavedModel (TF siap produksi)
tf.saved_model.save(model, 'semantic_faq_savedmodel')
print("semantic_faq_savedmodel/")

# Label mapping
with open('label_mapping.json', 'w') as f:
    json.dump({str(k): v for k, v in label_map.items()}, f, ensure_ascii=False, indent=2)
print("label_mapping.json")

for fname in ['semantic_faq_model.keras', 'label_mapping.json']:
    if os.path.exists(fname):
        print(f"   {fname} — {os.path.getsize(fname):,} bytes")
print("Semua model dan mapping berhasil disimpan!")

semantic_faq_model.keras
INFO:tensorflow:Assets written to: semantic_faq_savedmodel\assets


2026-05-30 10:10:44,952 [INFO] Assets written to: semantic_faq_savedmodel\assets


semantic_faq_savedmodel/
label_mapping.json
   semantic_faq_model.keras — 16,338,120 bytes
   label_mapping.json — 511 bytes
Semua model dan mapping berhasil disimpan!


<!-- # SECTION 8 — SETUP TIDB VECTOR Db
1. Membuat `config.env` (template)
2. Menghubungkan ke TiDB Cloud
3. Membuat tabel `knowledge` dengan kolom `VECTOR(1024)`
4. Meng-embed dataset dengan **BAAI/bge-m3** lalu insert ke TiDB (batch, resumable)

> ⚠️ **Isi `config.env` dengan kredensial TiDB kamu sebelum menjalankan cell berikutnya.** -->


In [22]:
from dotenv import load_dotenv
import os

load_dotenv("config.env", override=True)

print("DB_HOST =", os.getenv("DB_HOST"))
print("DB_PORT =", os.getenv("DB_PORT"))
print("DB_USER =", os.getenv("DB_USER"))
print("SSL_CA =", os.getenv("SSL_CA"))

DB_HOST = gateway01.ap-southeast-1.prod.aws.tidbcloud.com
DB_PORT = 4000
DB_USER = 3jozrxcgFyjN7ZP.root
SSL_CA = C:\Users\Erycaa\Downloads\caps\DBS-Capstone-2026\python\isrgrootx1.pem


In [23]:
import os
print(os.path.exists("isrgrootx1.pem"))

True


In [24]:
load_dotenv(
    r"C:\Users\Erycaa\Downloads\caps\DBS-Capstone-2026\config.env",
    override=True
)

True

In [25]:
import os
import mysql.connector
from dotenv import load_dotenv

load_dotenv()
connection = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", 4000)),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    ssl_verify_cert=True,
    ssl_verify_identity=True,
    use_pure=True
)

print("\nKoneksi berhasil ✔")


Koneksi berhasil ✔


In [26]:
# conn = connection
# cur = conn.cursor()

# cur.execute("""
# CREATE TABLE IF NOT EXISTS knowledge (
#     id INT AUTO_INCREMENT PRIMARY KEY,
#     soal TEXT,
#     jawaban TEXT,
#     contoh TEXT,
#     konteks TEXT,
#     topik VARCHAR(255),
#     subtopik VARCHAR(255),
#     jenis_pertanyaan VARCHAR(100),
#     kompleksitas VARCHAR(50),
#     content TEXT,
#     embedding VECTOR(1024)
# );
# """)

# conn.commit()
# print("Tabel knowledge berhasil dicek/dibuat")

# cur.close()

In [29]:
cur = connection.cursor()
cur.execute("USE chatbot_db")

cur.execute("""
SELECT id, topik, subtopik, soal
FROM knowledge
LIMIT 10
""")

print(cur.fetchall())

[(1, 'adaptasi makhluk hidup', 'adaptasi makhluk hidup', 'kemampuan makhluk hidup untuk menyesuaikan diri dengan lingkungannya disebut.'), (2, 'adaptasi makhluk hidup', 'adaptasi makhluk hidup', 'bentuk adaptasi yang berkaitan dengan fungsi organ internal adalah.'), (3, 'adaptasi makhluk hidup', 'adaptasi makhluk hidup', 'bentuk adaptasi yang berkaitan dengan bentuk tubuh dan struktur fisik adalah.'), (4, 'adaptasi makhluk hidup', 'adaptasi makhluk hidup', 'bentuk adaptasi yang berkaitan dengan perilaku dan kebiasaan hewan adalah.'), (5, 'adaptasi makhluk hidup', 'adaptasi makhluk hidup', 'hewan dengan paruh pipih yang digunakan menyaring makanan dari air adalah.'), (6, 'adaptasi makhluk hidup', 'adaptasi makhluk hidup', 'mengapa beruang kutub memiliki bulu tebal?'), (7, 'adaptasi makhluk hidup', 'adaptasi makhluk hidup', 'kenapa beruang kutub memiliki bulu tebal?'), (8, 'adaptasi makhluk hidup', 'adaptasi makhluk hidup', 'bagaimana beruang kutub memiliki bulu tebal?'), (9, 'adaptasi m

In [53]:
# Format 1: .keras 
model.save('semantic_faq_model.keras')
print("semantic_faq_model.keras")

tf.saved_model.save(model, 'semantic_faq_savedmodel')

with open('label_mapping.json', 'w') as f:
    json.dump({str(k): v for k, v in label_map.items()}, f, ensure_ascii=False, indent=2)

semantic_faq_model.keras
INFO:tensorflow:Assets written to: semantic_faq_savedmodel\assets


2026-05-30 11:12:10,393 [INFO] Assets written to: semantic_faq_savedmodel\assets


In [30]:
# Helper functions untuk embedding
def build_content(row):
    parts = [str(row.get(c, '')).strip() for c in ('soal','jawaban','contoh','konteks')]
    return ' '.join(p for p in parts if p and p.lower() != 'nan')

def format_embedding(emb):
    return '[' + ','.join(f'{v:.8f}' for v in emb) + ']'

def get_already_done(cursor):
    cursor.execute("SELECT COUNT(*) FROM knowledge")
    return cursor.fetchone()[0]

print("Helper functions siap digunaksn")


Helper functions siap digunaksn


In [31]:
from sentence_transformers import SentenceTransformer

print("Memuat model BAAI/bge-m3 ...")
embedder = SentenceTransformer("BAAI/bge-m3")
print("Model embedding bisa dipake")

2026-05-30 10:13:10,482 [INFO] Use pytorch device_name: cpu
2026-05-30 10:13:10,483 [INFO] Load pretrained SentenceTransformer: BAAI/bge-m3


Memuat model BAAI/bge-m3 ...


c:\Users\Erycaa\Downloads\caps\DBS-Capstone-2026\env\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model embedding bisa dipake


In [ ]:
# # Embed dataset & insert ke TiDB (batch, resumable)
# BATCH_SIZE = 32

# SQL_INSERT = """
#     INSERT INTO knowledge
#         (soal, jawaban, contoh, konteks,
#          topik, subtopik, jenis_pertanyaan, kompleksitas,
#          content, embedding)
#     VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
# """

# conn = mysql.connector.connect(**DB_CONFIG)
# cur  = conn.cursor()

# already_done = get_already_done(cur)
# df_remaining = data.iloc[already_done:].reset_index(drop=True)
# total_rows   = len(data)

# log.info("Sudah tersimpan: %d | Sisa: %d | Total: %d",
#          already_done, len(df_remaining), total_rows)

# if len(df_remaining) == 0:
#     print("Semua data sudah tersimpan di TiDB ✔")
# else:
#     total_ok = already_done
#     for batch_start in range(0, len(df_remaining), BATCH_SIZE):
#         batch    = df_remaining.iloc[batch_start:batch_start + BATCH_SIZE]
#         contents = [build_content(row) for _, row in batch.iterrows()]

#         end_idx = min(total_ok + len(batch), total_rows)
#         log.info("Embedding baris %d–%d dari %d ...", total_ok + 1, end_idx, total_rows)

#         embeddings = embedder.encode(
#             contents, batch_size=BATCH_SIZE,
#             normalize_embeddings=True, show_progress_bar=False
#         )

#         try:
#             conn.ping(reconnect=True, attempts=5, delay=3)
#             cur = conn.cursor()
#         except Exception as e:
#             log.warning("Ping gagal (%s), reconnecting ...", e)
#             conn = mysql.connector.connect(**DB_CONFIG)
#             cur  = conn.cursor()

#         for i, (_, row) in enumerate(batch.iterrows()):
#             try:
#                 cur.execute(SQL_INSERT, (
#                     row.get('soal',''), row.get('jawaban',''),
#                     row.get('contoh',''), row.get('konteks',''),
#                     row.get('topik',''), row.get('subtopik',''),
#                     row.get('jenis_pertanyaan',''), row.get('kompleksitas',''),
#                     contents[i], format_embedding(embeddings[i].tolist()),
#                 ))
#                 total_ok += 1
#             except Exception as e:
#                 log.error("Baris gagal: %s", e)

#         try:
#             conn.commit()
#             log.info("Batch di-commit (%d/%d)", total_ok, total_rows)
#         except Exception as e:
#             log.error("Commit gagal: %s", e)
#             conn = mysql.connector.connect(**DB_CONFIG)
#             cur  = conn.cursor()

#     cur.close()
#     conn.close()
#     print(f"\nSelesai! {total_ok}/{total_rows} baris tersimpan ke TiDB ✔")


In [32]:
cur = connection.cursor()

cur.execute("""
SELECT COUNT(*),
       COUNT(embedding)
FROM knowledge
""")

print(cur.fetchone())

cur.close()

(3348, 3348)


True

<!-- ## REST API

Buat file `app.py` yang bisa langsung dijalanin sebagai API.
Jalankan dengan: `uvicorn app:app --reload` -->

In [33]:
if not connection.is_connected():
    connection.reconnect(attempts=5, delay=2)

cur = connection.cursor()

cur.execute("SELECT COUNT(*) FROM knowledge")
total = cur.fetchone()[0]

cur.execute("SELECT DISTINCT topik FROM knowledge ORDER BY topik")
topics = [r[0] for r in cur.fetchall()]

cur.close()

print(f"Total baris : {total}")
print(f"Topik       : {len(topics)} kategori")

for t in topics:
    print(f" - {t}")

Total baris : 3348
Topik       : 15 kategori
 - adaptasi makhluk hidup
 - air
 - alat pencernaan dan makanan
 - alat pernapasan manusia dan hewan
 - alat tubuh manusia dan hewan
 - benda dan sifatnya
 - bumi dan peristiwa alam
 - cahaya dan sifat-sifatnya
 - gaya, gerak, dan energi
 - organ tubuh manusia dan hewan
 - peredaran darah
 - peristiwa alam
 - sistem pernapasan
 - sumber daya alam dan kegunaannya
 - tumbuhan hijau


<!-- # SECTION 9 — MODERATION SYSTEM 
Filter kata kasar dengan sistem **strike + cooldown 5 menit**. -->

In [34]:
def normalize_leet(text):
    leet = {'0':'o','1':'i','3':'e','4':'a','5':'s','7':'t','8':'b','@':'a','$':'s'}
    return ''.join(leet.get(c, c) for c in text)

def clean_text_mod(text):
    text = re.sub(r'[^\w\s]', '', text.lower().strip())
    return normalize_leet(re.sub(r'\s+', ' ', text))

def load_toxic_words(csv_path):
    words = set()
    try:
        with open(csv_path, 'r', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                w = row.get('text','').strip().lower()
                if w: words.add(w)
        log.info("Loaded %d kata toxic dari %s", len(words), csv_path)
    except FileNotFoundError:
        log.warning("File toxic tidak ditemukan: %s", csv_path)
    return words

print("Moderation utilities")


Moderation utilities


In [35]:
class ModerationSystem:
    """
    Sistem moderasi kata kasar untuk chatbot anak SD.

    Deteksi:
    1. Exact phrase match terhadap blacklist CSV
    2. Phrase containment (frasa >= 2 kata)
    3. Word-level match
    + normalisasi leet speak (b0d0h → bodoh)

    Strike system:
    - Strike 1 : warning ringan
    - Strike 2 : reminder sopan
    - Strike 3 : cooldown 5 menit
    """
    COOLDOWN_MINUTES = 5

    def __init__(self, toxic_csv='dataset/dataset_kata_kasar.csv'):
        self.toxic_phrases = load_toxic_words(toxic_csv)
        self.toxic_words   = {normalize_leet(w) for p in self.toxic_phrases
                              for w in p.split() if len(normalize_leet(w)) >= 3}
        log.info("Extracted %d kata toxic unik", len(self.toxic_words))
        self.sessions = {}

    def _get_session(self, sid):
        if sid not in self.sessions:
            self.sessions[sid] = {'count': 0, 'cooldown_until': None}
        return self.sessions[sid]

    def _is_on_cooldown(self, sid):
        s = self._get_session(sid)
        if s['cooldown_until'] is None: return False
        if datetime.now() >= s['cooldown_until']:
            s['count'] = 0; s['cooldown_until'] = None; return False
        return True

    def _is_toxic(self, text):
        c = clean_text_mod(text)
        if c in self.toxic_phrases: return {'is_toxic': True, 'matched': c}
        for p in self.toxic_phrases:
            if len(p.split()) >= 2 and p in c: return {'is_toxic': True, 'matched': p}
        for w in c.split():
            if normalize_leet(w) in self.toxic_words: return {'is_toxic': True, 'matched': normalize_leet(w)}
        return {'is_toxic': False, 'matched': None}

    def check(self, text, sid):
        if self._is_on_cooldown(sid):
            s   = self._get_session(sid)
            rem = max(1, (s['cooldown_until'] - datetime.now()).seconds // 60 + 1)
            return {'status':'cooldown','strikes':s['count'],'matched':None,
                    'message':f"Coba lagi dalam {rem} menit 🕐"}
        r = self._is_toxic(text)
        s = self._get_session(sid)
        if not r['is_toxic']:
            return {'status':'safe','strikes':s['count'],'message':None,'matched':None}
        s['count'] += 1; k = s['count']
        log.info("Session %s | toxic='%s' | strike=%d/3", sid, r['matched'], k)
        if k >= 3:
            s['cooldown_until'] = datetime.now() + timedelta(minutes=self.COOLDOWN_MINUTES)
            return {'status':'cooldown','strikes':k,'matched':r['matched'],
                    'message':f"Istirahat dulu {self.COOLDOWN_MINUTES} menit ya 🕐"}
        if k == 2:
            return {'status':'warning','strikes':2,'matched':r['matched'],
                    'message':"Sudah 2 peringatan. Yuk jaga kata-katanya ya 😊"}
        return {'status':'warning','strikes':1,'matched':r['matched'],
                'message':"Yuk gunakan bahasa yang lebih sopan ya 😊"}

    def reset_session(self, sid): self.sessions.pop(sid, None)
    def get_strikes(self, sid):   return self._get_session(sid)['count']

print("ModerationSystem")


ModerationSystem


In [36]:
import logging
import os
import csv
import re
from datetime import datetime, timedelta

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

toxic_path = r"DBS-Capstone-2026\dataset\dataset_kata_kasar.csv"

print("Cek file:", toxic_path)
print("Exists:", os.path.exists(toxic_path))


Cek file: DBS-Capstone-2026\dataset\dataset_kata_kasar.csv
Exists: False


In [37]:
mod_test = ModerationSystem()
test_cases = [
    ("Apa itu fotosintesis?", "safe"),
    ("dasar bodoh",           "strike 1"),
    ("goblok banget",         "strike 2"),
    ("anjing kamu",           "strike 3 cooldown"),
    ("Apa itu air?",          "cooldown aktif"),
]
print("=== Test Moderation System ===")
for text, expected in test_cases:
    r    = mod_test.check(text, 'test_mod')
    icon = '✅' if r['status']=='safe' else ('🕐' if r['status']=='cooldown' else '⚠️')
    print(f"{icon} [{expected:22s}] status={r['status']} | strikes={r['strikes']}")
    if r['message']: print(f"   💬 {r['message']}")


2026-05-30 10:14:51,179 [WARNING] File toxic tidak ditemukan: dataset/dataset_kata_kasar.csv
2026-05-30 10:14:51,180 [INFO] Extracted 0 kata toxic unik


=== Test Moderation System ===
✅ [safe                  ] status=safe | strikes=0
✅ [strike 1              ] status=safe | strikes=0
✅ [strike 2              ] status=safe | strikes=0
✅ [strike 3 cooldown     ] status=safe | strikes=0
✅ [cooldown aktif        ] status=safe | strikes=0


<!-- Section 10 Screen Time Manager
Melacak durasi belajra per sesi. Reminder di menit ke 20 dan 30 -->


In [38]:
ACTIVITY_SUGGESTIONS = [
    "🏃 Coba lari-lari kecil di halaman ya!",
    "📖 Baca buku cerita favoritmu 10 menit!",
    "🎨 Yuk gambar atau mewarnai sesuatu!",
    "🌳 Main di luar rumah sebentar!",
    "💪 Lakukan peregangan badan!",
    "🧋 Minum air putih dulu ya!",
    "👨\u200d👩\u200d👧 Ceritakan ke orang tua apa yang sudah kamu pelajari!",
    "🧩 Main puzzle edukatif!",
    "✍️ Tulis hal menarik di buku catatan!",
    "🎵 Dengarkan lagu favoritmu sambil istirahat!",
]

class ScreenTimeManager:
    """
    Melacak durasi belajar user & memberikan pengingat istirahat.
    Threshold: 20 menit (reminder ringan), 30 menit (reminder + saran aktivitas).
    """
    def __init__(self): self.sessions = {}

    def _get_session(self, sid):
        if sid not in self.sessions:
            self.sessions[sid] = {'start_time': datetime.now(), 'reminded_20': False, 'reminded_30': False}
            log.info("Session %s: timer dimulai.", sid)
        return self.sessions[sid]

    def check(self, sid):
        s = self._get_session(sid)
        d = (datetime.now() - s['start_time']).total_seconds() / 60
        r = {'duration_minutes': round(d,1), 'reminder': None, 'suggestion': None, 'should_break': False}
        if d >= 30 and not s['reminded_30']:
            s['reminded_30'] = True
            r.update({'reminder': "Sudah 30 menit belajar! 🌟 Istirahat sebentar ya.",
                      'suggestion': random.choice(ACTIVITY_SUGGESTIONS), 'should_break': True})
        elif d >= 20 and not s['reminded_20']:
            s['reminded_20'] = True
            r['reminder'] = "Sudah 20 menit! 📚 Sebentar lagi waktunya istirahat ya."
        return r

    def reset_session(self, sid): self.sessions.pop(sid, None)
    def get_duration(self, sid):
        s = self._get_session(sid)
        return (datetime.now() - s['start_time']).total_seconds() / 60

print("ScreenTimeManager ")


ScreenTimeManager 


In [40]:
import random

st_test = ScreenTimeManager()

for minutes, desc in [
    (0, 'Baru mulai'),
    (15, '15 menit'),
    (20, '20 menit'),
    (30, '30 menit')
]:
    sid = f"st_{minutes}"

    st_test._get_session(sid)
    st_test.sessions[sid]['start_time'] = datetime.now() - timedelta(minutes=minutes)

    r = st_test.check(sid)

    icon = '⏸️' if r['should_break'] else '✅'

    print(f"{icon} [{desc:12s}] durasi={r['duration_minutes']}m | break={r['should_break']}")

    if r.get('reminder'):
        print(f"   💬 {r['reminder']}")

    if r.get('suggestion'):
        print(f"   🌱 {r['suggestion']}")

    print("-" * 50)

2026-05-30 10:15:03,807 [INFO] Session st_0: timer dimulai.
2026-05-30 10:15:03,809 [INFO] Session st_15: timer dimulai.
2026-05-30 10:15:03,810 [INFO] Session st_20: timer dimulai.
2026-05-30 10:15:03,811 [INFO] Session st_30: timer dimulai.


✅ [Baru mulai  ] durasi=0.0m | break=False
--------------------------------------------------
✅ [15 menit    ] durasi=15.0m | break=False
--------------------------------------------------
✅ [20 menit    ] durasi=20.0m | break=False
   💬 Sudah 20 menit! 📚 Sebentar lagi waktunya istirahat ya.
--------------------------------------------------
⏸️ [30 menit    ] durasi=30.0m | break=True
   💬 Sudah 30 menit belajar! 🌟 Istirahat sebentar ya.
   🌱 💪 Lakukan peregangan badan!
--------------------------------------------------


<!-- RAG RETRIVER TiDB -->

In [41]:
import logging
from sentence_transformers import SentenceTransformer

log = logging.getLogger(__name__)


class RAGRetriever:
    """
    RAG Retrieval Engine menggunakan TiDB Vector Search.

    Alur:
    1. Embed query → vector (BAAI/bge-m3, 1024 dim)
    2. Cari dokumen paling mirip via VEC_COSINE_DISTANCE di TiDB
    3. Return jawaban + metadata

    Parameter threshold:
        Jarak cosine <= threshold dianggap relevan.
        (0 = identik, 1 = tidak mirip)
        Default 0.5 — sesuaikan berdasarkan kualitas dataset.
    """
    def __init__(self, db_connection=None):
        log.info("Memuat BAAI/bge-m3 ...")
        self.embedder = SentenceTransformer("BAAI/bge-m3")

        # pakai connection yang udah ada, atau fallback ke global `connection`
        self.db = db_connection or connection
        self.cursor = None
        self._buat_cursor()
        log.info("RAGRetriever siap ✔")

    def _buat_cursor(self):
        self.cursor = self.db.cursor(dictionary=True)

    def _ensure_connection(self):
        try:
            self.db.ping(reconnect=True, attempts=3, delay=2)
            self._buat_cursor()
        except Exception:
            log.warning("Koneksi terputus, reconnecting ...")
            self.db.reconnect(attempts=5, delay=2)
            self._buat_cursor()

    def retrieve(self, query, top_k=3):
        """Cari top_k dokumen paling relevan dari TiDB."""
        emb = self.embedder.encode(query, normalize_embeddings=True)
        emb_str = '[' + ','.join(f'{v:.8f}' for v in emb.tolist()) + ']'

        self._ensure_connection()
        self.cursor.execute("""
            SELECT soal, jawaban, topik, subtopik, contoh, konteks,
                   VEC_COSINE_DISTANCE(embedding, %s) AS distance
            FROM knowledge ORDER BY distance ASC LIMIT %s
        """, (emb_str, top_k))

        results = self.cursor.fetchall()
        for r in results:
            log.info("[%s] dist=%.4f | soal=%s", r['topik'], r['distance'], str(r['soal'])[:60])
        return results

    def get_best_answer(self, query, threshold=0.5):
        """Ambil jawaban terbaik. Jika tidak relevan, return fallback."""
        results = self.retrieve(query, top_k=1)

        if not results:
            return self._fallback()

        best = results[0]
        if best['distance'] > threshold:
            log.info("Tidak relevan | dist=%.4f > threshold=%.2f", best['distance'], threshold)
            return self._fallback()

        return {
            'answer'          : best['jawaban'],
            'category'        : best['topik'],
            'subtopik'        : best.get('subtopik', ''),
            'question_matched': best['soal'],
            'similarity_score': round(1 - best['distance'], 4),
            'contoh'          : best.get('contoh', ''),
            'konteks'         : best.get('konteks', ''),
        }

    def _fallback(self):
        return {
            'answer': (
                "Maaf, pertanyaan itu belum ada di pengetahuan saya 😊\n\n"
                "Coba tanyakan topik IPA seperti:\n"
                "- fotosintesis\n- pencernaan\n- adaptasi makhluk hidup\n- energi\n- gaya"
            ),
            'category': None, 'subtopik': '', 'question_matched': None,
            'similarity_score': 0.0, 'contoh': '', 'konteks': '',
        }

    def close(self):
        try:
            if self.cursor: self.cursor.close()
        except Exception: pass

print("RAGRetriever siap digunakan")

RAGRetriever siap digunakan


<!-- # ─── SECTION 12 — LOAD MODEL & INFERENCE ─────────────────────── -->

In [42]:
%pip uninstall protobuf -y
%pip install protobuf==4.25.3

Found existing installation: protobuf 4.25.9
Uninstalling protobuf-4.25.9:
  Successfully uninstalled protobuf-4.25.9
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


  Using cached protobuf-4.25.3-cp310-abi3-win_amd64.whl.metadata (541 bytes)
Using cached protobuf-4.25.3-cp310-abi3-win_amd64.whl (413 kB)
Note: you may need to restart the kernel to use updated packages.


In [45]:
# ── Load Model ──────────────────────────────────────────────────────

loaded_model = tf.keras.models.load_model(
    'semantic_faq_model.keras',
    custom_objects={
        'AttentionPooling': AttentionPooling,
        'FocalLoss': FocalLoss,
        'OneHotMAE': OneHotMAE
    }
)

with open('label_mapping.json') as f:
    loaded_label_map = {int(k): v for k, v in json.load(f).items()}

# pakai connection yang udah ada dari cell TiDB sebelumnya
retriever = RAGRetriever(db_connection=connection)

# quick check
test_pred = loaded_model.predict(
    tf.constant(['fotosintesis'], dtype=tf.string),
    verbose=0
)
print("model berhasil di-load ✔")
print("output shape :", test_pred.shape)
print("total prob   :", round(float(test_pred.sum()), 4), " harusnya sekitar 1.0")

2026-05-30 10:18:08,477 [INFO] Memuat BAAI/bge-m3 ...
2026-05-30 10:18:08,486 [INFO] Use pytorch device_name: cpu
2026-05-30 10:18:08,487 [INFO] Load pretrained SentenceTransformer: BAAI/bge-m3
c:\Users\Erycaa\Downloads\caps\DBS-Capstone-2026\env\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2026-05-30 10:18:24,422 [INFO] RAGRetriever siap ✔


model berhasil di-load ✔
output shape : (1, 15)
total prob   : 1.0  harusnya sekitar 1.0


In [46]:
def prediksi_topik(pertanyaan):
    """Prediksi topik IPA menggunakan TF classifier."""
    pred = loaded_model.predict(tf.constant([pertanyaan], dtype=tf.string), verbose=0)
    idx  = int(np.argmax(pred))
    return {'topic': loaded_label_map[idx], 'confidence': float(np.max(pred))}

def cari_jawaban(pertanyaan):
    """
    Pipeline inference lengkap:
      prediksi_topik (TF model) + retrieval jawaban (TiDB RAG)
    """
    topik  = prediksi_topik(pertanyaan)
    result = retriever.get_best_answer(pertanyaan)
    result['topik_prediksi'] = topik['topic']
    result['confidence']     = topik['confidence']
    result['pertanyaan']     = pertanyaan
    return result

print("Fungsi inference siap digunakan")


Fungsi inference siap digunakan


In [47]:
# ── Demo Inference ─────────────────────────────────────────
demo_q = [
    'fungsi cangkang kura kura',
    'apa itu pemantulan cahaya',
    'reaksi gelap pada fotosintesis',
    'cara kerja jantung manusia',
    'apa yang dimaksud gaya gravitasi',
]

print("=" * 70)
print("DEMO INFERENCE — CHATBOT IPA (TiDB RAG)")
print("=" * 70)

for q in demo_q:
    r = cari_jawaban(q)
    print(f"\n Pertanyaan   : {r['pertanyaan']}")
    print(f"   Topik TF model: {r['topik_prediksi']} ({r['confidence']:.1%})")
    print(f"   Topik TiDB    : {r.get('category', '-')}")
    print(f"   Subtopik      : {r.get('subtopik', '-')}")
    jawaban = r['answer']
    print(f"   Jawaban       : {jawaban[:120]}{'...' if len(jawaban)>120 else ''}")
    print(f"   Similarity    : {r['similarity_score']:.4f}")
    print("-" * 70)


DEMO INFERENCE — CHATBOT IPA (TiDB RAG)


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.62s/it]
2026-05-30 10:18:44,243 [INFO] [adaptasi makhluk hidup] dist=0.2921 | soal=apa fungsi cangkang pada kura-kura?



 Pertanyaan   : fungsi cangkang kura kura
   Topik TF model: adaptasi makhluk hidup (84.8%)
   Topik TiDB    : adaptasi makhluk hidup
   Subtopik      : adaptasi makhluk hidup
   Jawaban       : cangkang kura-kura berfungsi melindungi tubuhnya dari musuh dan benturan.
   Similarity    : 0.7079
----------------------------------------------------------------------


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.31it/s]
2026-05-30 10:18:45,042 [INFO] [cahaya dan sifat-sifatnya] dist=0.2604 | soal=apa yang dimaksud pemantulan cahaya?



 Pertanyaan   : apa itu pemantulan cahaya
   Topik TF model: cahaya dan sifat-sifatnya (99.7%)
   Topik TiDB    : cahaya dan sifat-sifatnya
   Subtopik      : sifat cahaya pada berbagai benda
   Jawaban       : pemantulan cahaya adalah peristiwa ketika cahaya memantul kembali setelah mengenai permukaan benda.
   Similarity    : 0.7396
----------------------------------------------------------------------


...


 Pertanyaan   : reaksi gelap pada fotosintesis
   Topik TF model: tumbuhan hijau (100.0%)
   Topik TiDB    : tumbuhan hijau
   Subtopik      : fotosintesis
   Jawaban       : fase gelap adalah tahap fotosintesis yang berlangsung tanpa cahaya matahari secara langsung.
   Similarity    : 0.7294
----------------------------------------------------------------------


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.20it/s]
2026-05-30 10:18:46,657 [INFO] [alat tubuh manusia dan hewan] dist=0.3052 | soal=jantung kita kan berdetak. setiap detakan itu pompa darah. g



 Pertanyaan   : cara kerja jantung manusia
   Topik TF model: peredaran darah (63.6%)
   Topik TiDB    : alat tubuh manusia dan hewan
   Subtopik      : sistem peredaran darah
   Jawaban       : setiap detakan jantung memompa darah ke seluruh tubuh melalui pembuluh darah.
   Similarity    : 0.6948
----------------------------------------------------------------------


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.17it/s]
2026-05-30 10:18:47,776 [INFO] [gaya, gerak, dan energi] dist=0.3043 | soal=apa yang dimaksud dengan gaya?



 Pertanyaan   : apa yang dimaksud gaya gravitasi
   Topik TF model: gaya, gerak, dan energi (99.8%)
   Topik TiDB    : gaya, gerak, dan energi
   Subtopik      : pengertian gaya
   Jawaban       : gaya adalah dorongan atau tarikan yang mempengaruhi benda.
   Similarity    : 0.6957
----------------------------------------------------------------------


<!-- # ─── SECTION 13 — REST API (FastAPI) ────────────────────────────

| Endpoint | Method | Keterangan |
|---|---|---|
| `/` | GET | Info API |
| `/health` | GET | Status semua komponen |
| `/chat` | POST | Chatbot utama (RAG TiDB + Moderasi + Screen Time) |
| `/moderation` | POST | Cek teks toxic saja |
| `/docs` | GET | Swagger UI interaktif |

Jalankan: `uvicorn app:app --host 0.0.0.0 --port 8000 --reload` -->


In [54]:
if os.path.exists('app.py'):
    print(f"✔ app.py berhasil dibuat ({os.path.getsize('app.py'):,} bytes)")
    print()
    print("Cara menjalankan:  uvicorn app:app --host 0.0.0.0 --port 8000 --reload")
    print()
    print("Endpoint:")
    print("  GET  http://localhost:8000/health")
    print("  POST http://localhost:8000/chat        chatbot utama")
    print("  POST http://localhost:8000/moderation  cek toxic")
    print("  GET  http://localhost:8000/docs        Swagger UI")
else:
    print("✘ Gagal membuat app.py")


✔ app.py berhasil dibuat (14,375 bytes)

Cara menjalankan:  uvicorn app:app --host 0.0.0.0 --port 8000 --reload

Endpoint:
  GET  http://localhost:8000/health
  POST http://localhost:8000/chat        chatbot utama
  POST http://localhost:8000/moderation  cek toxic
  GET  http://localhost:8000/docs        Swagger UI


<!-- ## Test API -->

In [55]:
import importlib, sys, threading, time
import nest_asyncio, uvicorn
import requests

nest_asyncio.apply()

def run_server():
    if 'app' in sys.modules:
        del sys.modules['app']
    m = importlib.import_module('app')
    uvicorn.run(m.app, host='127.0.0.1', port=8000, log_level='warning')

threading.Thread(target=run_server, daemon=True).start()

# Tungguin server siap
print("Menunggu server siap", end="")
for _ in range(60):  # maksimal 60 detik
    try:
        requests.get("http://127.0.0.1:8000/health", timeout=1)
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(1)

print()
print('✔ Server berjalan di http://127.0.0.1:8000')
print('📖 Swagger UI: http://127.0.0.1:8000/docs')

Menunggu server siap
✔ Server berjalan di http://127.0.0.1:8000
📖 Swagger UI: http://127.0.0.1:8000/docs
Loading model TensorFlow ...


In [56]:
import requests

BASE = 'http://127.0.0.1:8000'

# /health
print('=== /health ===')
try:
    h = requests.get(f'{BASE}/health', timeout=30).json()
    print(f"  Status: {h['status']}")
    for k, v in h['components'].items():
        print(f'  {k}: {v}')
except Exception as e:
    print(f'  ❌ Error: {e}')

# /chat
print('\n=== /chat (pertanyaan normal) ===')
try:
    r = requests.post(f'{BASE}/chat',
        json={'message': 'Apa itu fotosintesis?', 'session_id': 'notebook_test'},
        timeout=60).json()
    print(f"  Topik TiDB   : {r.get('category', '-')}")
    print(f"  Topik TF     : {r.get('predicted_topic', '-')} ({r.get('tf_confidence', 0):.1%})")
    print(f"  Similarity   : {r.get('similarity_score', 0):.4f}")
    ans = r.get('answer', '-')
    print(f"  Jawaban      : {ans[:120]}{'...' if len(ans)>120 else ''}")
    print(f"  Moderasi     : {r.get('moderation', {}).get('status', '-')}")
    print(f"  Screen Time  : {r.get('screen_time', {}).get('duration_minutes', '-')} menit")
except Exception as e:
    print(f'  ❌ Error: {e}')

# /moderation
print('\n=== /moderation (kata kasar) ===')
try:
    r2 = requests.post(f'{BASE}/moderation',
        json={'text': 'dasar bodoh', 'session_id': 'notebook_mod'},
        timeout=30).json()
    print(f"  Status  : {r2.get('status', '-')}")
    print(f"  Strikes : {r2.get('strikes', '-')}")
    print(f"  Pesan   : {r2.get('message', '-')}")
except Exception as e:
    print(f'  ❌ Error: {e}')

print('\nTest selesai!')


=== /health ===
  Status: ok
  tf_model: loaded
  tidb: connected
  moderation: active
  screen_time: active

=== /chat (pertanyaan normal) ===
  Topik TiDB   : tumbuhan hijau
  Topik TF     : tumbuhan hijau (100.0%)
  Similarity   : 0.7481
  Jawaban      : fotosintesis adalah proses membuat makanan dengan cahaya matahari.
  Moderasi     : safe
  Screen Time  : 0.0 menit

=== /moderation (kata kasar) ===
  Status  : warning
  Strikes : 1
  Pesan   : Yuk gunakan bahasa yang lebih sopan ya 😊

Test selesai!


In [ ]:
%pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


: 

In [ ]:
%pip install pyngrok -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import importlib, sys, threading, time
import nest_asyncio, uvicorn
from pyngrok import ngrok, conf

NGROK_AUTHTOKEN = "3ELabcoLKg1JsLqH71UGEH9U4WG_4bt1sP2uNRTMW5aqY2hXQ"

conf.get_default().auth_token = NGROK_AUTHTOKEN
nest_asyncio.apply()

def run_server():
    if 'app' in sys.modules:
        del sys.modules['app']
    m = importlib.import_module('app')
    uvicorn.run(m.app, host='127.0.0.1', port=8000, log_level='warning')

# jalankan server di background
threading.Thread(target=run_server, daemon=True).start()
time.sleep(8)  # tunggu server siap

# buka tunnel ngrok
public_url = ngrok.connect(8000)

print('=' * 60)
print('Server jalan + Ngrok tunnel aktif')
print()
print('URL PUBLIK (fullstack):')
print(f'   {public_url}')
print()
print('Swagger UI:')
print(f'   {public_url}/docs')
print()
print('Endpoint yang tersedia:')
print(f'   GET  {public_url}/health')
print(f'   POST {public_url}/chat')
print(f'   POST {public_url}/moderation')
print('=' * 60)

Loading model TensorFlow ...


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-m3
INFO:pyngrok.ngrok:Opening tunnel named: http-8000-7c323b7e-193f-4597-a115-b815d80bcd2e


INFO:pyngrok.process:Overriding default auth token


Semua komponen berhasil di-load ✔


ERROR:    [Errno 10048] error while attempting to bind on address ('127.0.0.1', 8000): only one usage of each socket address (protocol/network address/port) is normally permitted
INFO:pyngrok.process.ngrok:t=2026-05-28T16:27:15+0700 lvl=info msg="no configuration paths supplied"
INFO:pyngrok.process.ngrok:t=2026-05-28T16:27:15+0700 lvl=info msg="using configuration at default config path" path=C:\\Users\\Erycaa\\AppData\\Local\\ngrok\\ngrok.yml
INFO:pyngrok.process.ngrok:t=2026-05-28T16:27:15+0700 lvl=info msg="open config file" path=C:\\Users\\Erycaa\\AppData\\Local\\ngrok\\ngrok.yml err=nil
INFO:pyngrok.process.ngrok:t=2026-05-28T16:27:15+0700 lvl=info msg="FIPS 140 mode" enabled=false
INFO:pyngrok.process.ngrok:t=2026-05-28T16:27:15+0700 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
INFO:pyngrok.process.ngrok:t=2026-05-28T16:27:16+0700 lvl=info msg="client session established" obj=tunnels.session
INFO:pyngrok.process.ngrok:t=2026-05-28T16:27:16+0700 

Server jalan + Ngrok tunnel aktif

URL PUBLIK (fullstack):
   NgrokTunnel: "https://groin-multitude-earphone.ngrok-free.dev" -> "http://localhost:8000"

Swagger UI:
   NgrokTunnel: "https://groin-multitude-earphone.ngrok-free.dev" -> "http://localhost:8000"/docs

Endpoint yang tersedia:
   GET  NgrokTunnel: "https://groin-multitude-earphone.ngrok-free.dev" -> "http://localhost:8000"/health
   POST NgrokTunnel: "https://groin-multitude-earphone.ngrok-free.dev" -> "http://localhost:8000"/chat
   POST NgrokTunnel: "https://groin-multitude-earphone.ngrok-free.dev" -> "http://localhost:8000"/moderation


INFO:pyngrok.process.ngrok:t=2026-05-28T16:27:17+0700 lvl=info msg=end pg=/api/tunnels id=bff1d64aa1a80be2 status=201 dur=882.2453ms


INFO:pyngrok.process.ngrok:t=2026-05-28T16:28:01+0700 lvl=info msg="join connections" obj=join id=9bd77eb87745 l=127.0.0.1:8000 r=114.124.247.215:56180
INFO:pyngrok.process.ngrok:t=2026-05-28T16:28:10+0700 lvl=info msg="join connections" obj=join id=167ca3558c03 l=127.0.0.1:8000 r=114.124.247.215:62828
INFO:pyngrok.process.ngrok:t=2026-05-28T16:31:11+0700 lvl=info msg="join connections" obj=join id=9a5604cfff8a l=127.0.0.1:8000 r=114.124.247.215:56817
INFO:pyngrok.process.ngrok:t=2026-05-28T16:31:12+0700 lvl=info msg="join connections" obj=join id=f3404bb1886d l=127.0.0.1:8000 r=114.124.247.215:56820
INFO:pyngrok.process.ngrok:t=2026-05-28T16:32:12+0700 lvl=info msg="join connections" obj=join id=ce0e069b47b9 l=127.0.0.1:8000 r=114.124.247.215:62828
INFO:pyngrok.process.ngrok:t=2026-05-28T16:32:18+0700 lvl=info msg="join connections" obj=join id=de11797180ca l=127.0.0.1:8000 r=114.124.241.199:54928
ERROR:pyngrok.process.ngrok:t=2026-05-28T16:48:16+0700 lvl=eror msg="heartbeat timeout, 

In [ ]:
import subprocess
# cari proses yang pakai port 8000 lalu kill
subprocess.run('netstat -ano | findstr :8000', shell=True)

CompletedProcess(args='netstat -ano | findstr :8000', returncode=0)

In [ ]:
import requests

# ambil URL string yang bersih dari object NgrokTunnel
BASE = public_url.public_url  # ← ambil attribute .public_url

print('BASE URL:', BASE)

print('=== Test /health ===')
h = requests.get(f'{BASE}/health', timeout=30).json()
print('Status    :', h['status'])
print('Komponen  :', h['components'])

print()
print('=== Test /chat ===')
r = requests.post(f'{BASE}/chat', json={
    'message': 'apa itu fotosintesis?',
    'session_id': 'test_deploy'
}, timeout=60).json()
print('Topik     :', r.get('predicted_topic'))
print('Similarity:', r.get('similarity_score'))
jawaban = r.get('answer', '')
print('Jawaban   :', jawaban[:120], '...' if len(jawaban) > 120 else '')

print()
print('✔ API berjalan normal dan bisa diakses publik!')

BASE URL: https://groin-multitude-earphone.ngrok-free.dev
=== Test /health ===
Status    : ok
Komponen  : {'tf_model': 'loaded', 'tidb': 'connected', 'moderation': 'active', 'screen_time': 'active'}

=== Test /chat ===


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.46it/s]


Topik     : benda dan sifatnya
Similarity: 0.749
Jawaban   : fotosintesis adalah proses membuat makanan dengan cahaya matahari. 

✔ API berjalan normal dan bisa diakses publik!


In [ ]:
# # matikan tunnel ngrok kalau sudah selesai
# ngrok.disconnect(public_url)
# ngrok.kill()
# print('Tunnel ngrok dimatikan.')